In [18]:
import os
import glob
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader, random_split

Loading in Data

In [19]:
try:
    import tifffile
    TIFFFILE_AVAILABLE = True
except ImportError:
    TIFFFILE_AVAILABLE = False

In [20]:
# -----------------------------------
# 1. File loading helpers
# -----------------------------------

def load_tif_image(path: str) -> np.ndarray:
    if TIFFFILE_AVAILABLE:
        try:
            img = tifffile.imread(path)
        except Exception:
            img = np.array(Image.open(path))
    else:
        img = np.array(Image.open(path))

    img = np.asarray(img)

    if img.ndim == 2:
        pass
    elif img.ndim == 3:
        if img.shape[0] < min(img.shape[1], img.shape[2]):
            img = img.mean(axis=0)
        else:
            img = img.mean(axis=-1)
    else:
        raise ValueError(f"Unsupported image shape {img.shape} for file {path}")

    return img.astype(np.float32)


def load_mask_image(path: str) -> np.ndarray:
    mask = np.array(Image.open(path))
    if mask.ndim == 3:
        mask = mask[..., 0]
    return (mask > 0).astype(np.float32)

In [21]:
# -----------------------------------
# 2. Preprocessing helpers
# -----------------------------------

def percentile_normalize(img: np.ndarray, lower: float = 1, upper: float = 99) -> np.ndarray:
    lo = np.percentile(img, lower)
    hi = np.percentile(img, upper)

    if hi <= lo:
        img = img - img.min()
        denom = img.max()
        if denom > 0:
            img = img / denom
        return img.astype(np.float32)

    img = np.clip(img, lo, hi)
    img = (img - lo) / (hi - lo)
    return img.astype(np.float32)


def resize_image_and_mask(image: np.ndarray, mask: np.ndarray, size=(256, 256)):
    image_pil = Image.fromarray((image * 255).astype(np.uint8))
    mask_pil = Image.fromarray((mask * 255).astype(np.uint8))

    image_resized = image_pil.resize(size, Image.BILINEAR)
    mask_resized = mask_pil.resize(size, Image.NEAREST)

    image_out = np.array(image_resized).astype(np.float32) / 255.0
    mask_out = (np.array(mask_resized) > 0).astype(np.float32)

    return image_out, mask_out

In [26]:
# -----------------------------------
# 3. CSV parsing + prefix extraction
# -----------------------------------

def load_counts_csv(count_csv_path: str) -> pd.DataFrame:
    """
    Load the counts CSV correctly.

    The file has 4 real columns, but each row also has a trailing comma.
    So we explicitly keep only the first 4 columns.
    """
    df = pd.read_csv(
        count_csv_path,
        index_col=False,
        usecols=[0, 1, 2, 3]
    )

    # Clean whitespace
    df.columns = [c.strip() for c in df.columns]

    required = ["Image_Count_Nuclei", "Image_FileName_OrigDAPI"]
    for col in required:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    df["Image_FileName_OrigDAPI"] = df["Image_FileName_OrigDAPI"].astype(str).str.strip()
    df["Image_Count_Nuclei"] = pd.to_numeric(df["Image_Count_Nuclei"], errors="coerce")

    df = df.dropna(subset=["Image_Count_Nuclei", "Image_FileName_OrigDAPI"]).copy()
    df["Image_Count_Nuclei"] = df["Image_Count_Nuclei"].astype(int)

    return df


def get_prefix_from_raw_filename(raw_filename: str) -> str:
    """
    Example:
    mcf-z-stacks-03212011_a01_s1_w29fd6a02-....tif
    -> mcf-z-stacks-03212011_a01_s1
    """
    stem = Path(raw_filename).stem
    if "_w" not in stem:
        raise ValueError(f"Filename does not contain '_w': {raw_filename}")
    return stem.split("_w")[0]

In [27]:
df = load_counts_csv("BBBC006_v1_counts.csv")
print(df.head())
print("rows:", len(df))
print(df.columns.tolist())

   Image_Count_Nuclei                            Image_FileName_OrigDAPI  \
0                   7  mcf-z-stacks-03212011_a01_s1_w197083f74-b742-4...   
1                  18  mcf-z-stacks-03212011_a01_s2_w1a63ef720-8648-4...   
2                  68  mcf-z-stacks-03212011_a02_s1_w1ebe36734-a690-4...   
3                  80  mcf-z-stacks-03212011_a02_s2_w1376ffd1a-e96b-4...   
4                  11  mcf-z-stacks-03212011_a03_s1_w1554aa323-1f38-4...   

   Image_Metadata_Site Image_Metadata_Well  
0                    1                 a01  
1                    2                 a01  
2                    1                 a02  
3                    2                 a02  
4                    1                 a03  
rows: 768
['Image_Count_Nuclei', 'Image_FileName_OrigDAPI', 'Image_Metadata_Site', 'Image_Metadata_Well']


In [23]:
# -----------------------------------
# 4. Dataset
# -----------------------------------

class NucleiSegmentationDataset(Dataset):
    def __init__(
        self,
        raw_dir: str,
        mask_dir: str,
        count_csv_path: str,
        image_size=(256, 256),
        return_original_size: bool = False,
        verbose: bool = True
    ):
        self.raw_dir = Path(raw_dir)
        self.mask_dir = Path(mask_dir)
        self.image_size = image_size
        self.return_original_size = return_original_size
        self.verbose = verbose

        self.df = load_counts_csv(count_csv_path)
        self.samples = self._build_samples()

        if len(self.samples) == 0:
            raise ValueError("No valid raw/mask/count matches found.")

    def _build_samples(self):
        samples = []

        # Build raw prefix -> list of candidate raw files
        raw_files = sorted(list(self.raw_dir.glob("*.tif")) + list(self.raw_dir.glob("*.tiff")))
        raw_prefix_to_files = {}

        for raw_path in raw_files:
            try:
                prefix = get_prefix_from_raw_filename(raw_path.name)
            except ValueError:
                continue
            raw_prefix_to_files.setdefault(prefix, []).append(raw_path)

        # Build mask lookup by stem
        mask_files = sorted(self.mask_dir.glob("*.png"))
        mask_stem_to_path = {m.stem: m for m in mask_files}

        unmatched_csv = 0
        unmatched_mask = 0
        unmatched_raw = 0

        for _, row in self.df.iterrows():
            raw_filename_from_csv = row["Image_FileName_OrigDAPI"]
            count = int(row["Image_Count_Nuclei"])

            try:
                prefix = get_prefix_from_raw_filename(raw_filename_from_csv)
            except ValueError:
                unmatched_csv += 1
                continue

            # mask should be prefix + .png
            mask_path = mask_stem_to_path.get(prefix, None)
            if mask_path is None:
                unmatched_mask += 1
                continue

            candidate_raws = raw_prefix_to_files.get(prefix, [])
            if len(candidate_raws) == 0:
                unmatched_raw += 1
                continue

            # Temporary rule: choose first candidate deterministically
            chosen_raw = sorted(candidate_raws)[0]

            samples.append({
                "raw_path": str(chosen_raw),
                "mask_path": str(mask_path),
                "count": count,
                "prefix": prefix,
                "raw_filename": chosen_raw.name,
                "mask_filename": mask_path.name
            })

        if self.verbose:
            print(f"CSV rows: {len(self.df)}")
            print(f"Raw prefixes found: {len(raw_prefix_to_files)}")
            print(f"Mask files found: {len(mask_files)}")
            print(f"Matched samples: {len(samples)}")
            print(f"Skipped rows with bad CSV filename format: {unmatched_csv}")
            print(f"Skipped rows with missing mask: {unmatched_mask}")
            print(f"Skipped rows with missing raw image: {unmatched_raw}")

            # show a few examples
            for s in samples[:5]:
                print(f"MATCH: {s['prefix']} -> raw={s['raw_filename']} | mask={s['mask_filename']} | count={s['count']}")

        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        image = load_tif_image(sample["raw_path"])
        mask = load_mask_image(sample["mask_path"])

        if image.shape != mask.shape:
            raise ValueError(
                f"Shape mismatch for {sample['prefix']}: image {image.shape}, mask {mask.shape}"
            )

        original_size = image.shape

        image = percentile_normalize(image)
        image, mask = resize_image_and_mask(image, mask, self.image_size)

        image_tensor = torch.from_numpy(image).unsqueeze(0).float()
        mask_tensor = torch.from_numpy(mask).unsqueeze(0).float()
        count_tensor = torch.tensor(sample["count"], dtype=torch.float32)

        out = {
            "image": image_tensor,
            "mask": mask_tensor,
            "count": count_tensor,
            "image_name": sample["raw_filename"],
            "mask_name": sample["mask_filename"],
            "prefix": sample["prefix"]
        }

        if self.return_original_size:
            out["original_size"] = original_size

        return out

In [24]:
# -----------------------------------
# 5. Split helpers
# -----------------------------------

def create_data_splits(
    raw_dir: str,
    mask_dir: str,
    count_csv_path: str,
    image_size=(256, 256),
    train_ratio: float = 0.7,
    val_ratio: float = 0.15,
    test_ratio: float = 0.15,
    seed: int = 42
):
    if abs(train_ratio + val_ratio + test_ratio - 1.0) > 1e-8:
        raise ValueError("train_ratio + val_ratio + test_ratio must sum to 1.")

    full_dataset = NucleiSegmentationDataset(
        raw_dir=raw_dir,
        mask_dir=mask_dir,
        count_csv_path=count_csv_path,
        image_size=image_size,
        verbose=True
    )

    n_total = len(full_dataset)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)
    n_test = n_total - n_train - n_val

    generator = torch.Generator().manual_seed(seed)

    train_dataset, val_dataset, test_dataset = random_split(
        full_dataset,
        [n_train, n_val, n_test],
        generator=generator
    )

    return full_dataset, train_dataset, val_dataset, test_dataset


def create_dataloaders(train_dataset, val_dataset, test_dataset, batch_size=8, num_workers=0):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    return train_loader, val_loader, test_loader

In [28]:
raw_dir = "raw_images"
mask_dir = "labeled_images"
count_csv_path = "BBBC006_v1_counts.csv"

full_dataset, train_dataset, val_dataset, test_dataset = create_data_splits(
    raw_dir=raw_dir,
    mask_dir=mask_dir,
    count_csv_path=count_csv_path,
    image_size=(256, 256),
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42
)

print("Total matched samples:", len(full_dataset))
print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

CSV rows: 768
Raw prefixes found: 4
Mask files found: 4
Matched samples: 4
Skipped rows with bad CSV filename format: 0
Skipped rows with missing mask: 764
Skipped rows with missing raw image: 0
MATCH: mcf-z-stacks-03212011_a01_s1 -> raw=mcf-z-stacks-03212011_a01_s1_w1248254b0-0193-4e11-8762-62b5d2b86216.tif | mask=mcf-z-stacks-03212011_a01_s1.png | count=7
MATCH: mcf-z-stacks-03212011_a01_s2 -> raw=mcf-z-stacks-03212011_a01_s2_w16103c57d-a39e-468b-bb15-d55443c280d9.tif | mask=mcf-z-stacks-03212011_a01_s2.png | count=18
MATCH: mcf-z-stacks-03212011_a10_s1 -> raw=mcf-z-stacks-03212011_a10_s1_w156636015-b90a-4755-b5e9-4ad5c11a244a.tif | mask=mcf-z-stacks-03212011_a10_s1.png | count=31
MATCH: mcf-z-stacks-03212011_a10_s2 -> raw=mcf-z-stacks-03212011_a10_s2_w1892c2650-bdc9-4a35-b3a3-082eab4aed84.tif | mask=mcf-z-stacks-03212011_a10_s2.png | count=30
Total matched samples: 4
Train: 2
Val: 0
Test: 2
